In [6]:
# =========================
# 0) Setup e imports - Competencia Data Mining UBA 2025
# =========================
!pip -q install polars==1.7.1 lightgbm==4.5.0 optuna==3.6.1 matplotlib seaborn

import polars as pl
from pathlib import Path
import json, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import optuna

# Seeds oficiales de la competencia
SEEDS = [509963, 739373, 794341, 900623, 917827]
# Semillas adicionales (números primos grandes)
SEEDS_EXTRA = [1299827, 1500007, 1700021, 1900009, 2100013]
SEEDS_ALL = SEEDS + SEEDS_EXTRA  # 10 semillas totales

# Paths definitivos en Google Cloud / repo git
CSV_PATH    = Path("competencia_02_crudo.csv")
SCHEMA_PATH = Path("schema.json")
BASE_DIR    = Path("dmeyf2025")
# SUBMIT_PATH = BASE_DIR / "submit_202106_v10.csv"  # Definir una vez confirmado el destino

# Configuración de la competencia
TRAIN_MESES = [201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102]  # Meses de entrenamiento
APPLY_MES   = 202104                            # Mes de aplicación


In [7]:
# =========================
# 1) Lectura de DF con Schema
# =========================

# Cargar schema
with open(SCHEMA_PATH) as f:
    schema_dict = json.load(f)

# Convertir strings a dtypes de Polars
schema = {k: getattr(pl, v) for k, v in schema_dict.items()}

# Lectura estricta (sin inferir tipos)
df = pl.read_csv(CSV_PATH, schema_overrides=schema)
df = df.rechunk()  # Compactar memoria

In [8]:
# =========================
# 2) Construcción de clase_ternaria (baseline del profesor)
# =========================

def agregar_clase_ternaria_optimizada(df: pl.DataFrame) -> pl.DataFrame:
    """
    Versión optimizada usando Polars - Baseline del profesor
    """
    return (
        df.lazy()
        .sort(["numero_de_cliente", "foto_mes"])
        .with_columns([
            pl.when(
                pl.col("foto_mes").shift(-1).over("numero_de_cliente").is_null() &
                (pl.col("foto_mes") != pl.col("foto_mes").max())
            ).then(pl.lit("BAJA+1"))
            .when(
                pl.col("foto_mes").shift(-2).over("numero_de_cliente").is_null() &
                (pl.col("foto_mes") <= pl.col("foto_mes").max() - 2)
            ).then(pl.lit("BAJA+2"))
            .otherwise(pl.lit("CONTINUA"))
            .alias("clase_ternaria")
        ])
        .collect()
    )

# Agregar la columna clase_ternaria
df = agregar_clase_ternaria_optimizada(df)

# Tabla de frecuencia de clase_ternaria por foto_mes
tabla_clases = (
    df.pivot(
        values="numero_de_cliente",
        index="foto_mes",
        on="clase_ternaria",
        aggregate_function="len"
    )
    .fill_null(0)
)

print(tabla_clases)


shape: (32, 4)
┌──────────┬──────────┬────────┬────────┐
│ foto_mes ┆ CONTINUA ┆ BAJA+2 ┆ BAJA+1 │
│ ---      ┆ ---      ┆ ---    ┆ ---    │
│ i64      ┆ u32      ┆ u32    ┆ u32    │
╞══════════╪══════════╪════════╪════════╡
│ 201901   ┆ 122918   ┆ 720    ┆ 635    │
│ 201902   ┆ 123985   ┆ 693    ┆ 723    │
│ 201903   ┆ 124535   ┆ 738    ┆ 694    │
│ 201904   ┆ 125293   ┆ 502    ┆ 743    │
│ 201905   ┆ 126016   ┆ 681    ┆ 505    │
│ …        ┆ …        ┆ …      ┆ …      │
│ 202104   ┆ 161340   ┆ 1126   ┆ 952    │
│ 202105   ┆ 161946   ┆ 842    ┆ 1129   │
│ 202106   ┆ 162336   ┆ 1135   ┆ 842    │
│ 202107   ┆ 163459   ┆ 0      ┆ 1137   │
│ 202108   ┆ 164822   ┆ 0      ┆ 0      │
└──────────┴──────────┴────────┴────────┘


In [9]:
# =========================
# 3) Feature Engineering Histórico - Lags y DeltaLags (baseline del profesor)
# =========================

# Excluir IDs y metadata
EXCLUDE = {"numero_de_cliente", "foto_mes", "clase_ternaria"}

# Identificar columnas numéricas
num_cols = [c for c, dt in zip(df.columns, df.dtypes)
            if c not in EXCLUDE and dt.is_numeric()]

def add_lags_and_deltas(df: pl.DataFrame, cols: list[str]) -> pl.DataFrame:
    """
    Agregar lags y deltas de orden 1 y 2 - Baseline del profesor
    """
    out = df.sort(["numero_de_cliente", "foto_mes"]).lazy()

    # Lags 1 y 2
    out = out.with_columns([
        pl.col(c).shift(1).over("numero_de_cliente").alias(f"{c}_lag1") for c in cols
    ] + [
        pl.col(c).shift(2).over("numero_de_cliente").alias(f"{c}_lag2") for c in cols
    ])

    # Delta 1: x_t - x_{t-1} ; Delta 2: x_t - x_{t-2}
    out = out.with_columns([
        (pl.col(c) - pl.col(f"{c}_lag1")).alias(f"{c}_d1") for c in cols
    ] + [
        (pl.col(c) - pl.col(f"{c}_lag2")).alias(f"{c}_d2") for c in cols
    ])

    return out.collect()

# Aplicar lags y deltas
df = add_lags_and_deltas(df, num_cols)
